# B4. Làm sạch dữ liệu

Hai phần:
1. **Kiểm tra** (chỉ trên khách tập train): trùng, mã đặc biệt, giá trị không thể xảy ra, mâu thuẫn, ngoại lai. Mỗi dòng: số dòng, tỷ lệ, cách xử lý.
2. **Làm sạch và ghi ra** `ml/data/clean/<bảng>.parquet` cho cả 7 bảng, mọi dòng. Các notebook sau đọc từ đây.

Chỉ sửa hai loại, bằng **quy tắc cố định** (không tính trung bình, phân vị hay bất kỳ thống kê nào từ dữ liệu), nên áp dụng cho cả test không làm lộ thông tin test:
- Mã đặc biệt → trống: `365243` ở cột `DAYS_` (Home Credit: "vô cùng"), chữ `XNA`/`XAP` (Home Credit: giá trị trống), `NAME_FAMILY_STATUS` = Unknown. Mã mang thông tin thì thêm cờ (`FLAG_NOT_EMPLOYED`, `FLAG_NOT_TERMINATED`).
- Ngày không thể ở `bureau` → trống (giữ dòng).

Trùng, mâu thuẫn, ngoại lai: **giữ nguyên**, chỉ báo cáo tỷ lệ. Không xóa dòng nào, không chặn giá trị nào.
Ý nghĩa các mã: câu trả lời của Home Credit trên diễn đàn Kaggle của cuộc thi.

In [1]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
RAW_DIR = ROOT / "data" / "ml" / "home_credit"
DATA_DIR = ROOT / "ml" / "data"

import numpy as np
import pandas as pd

CLEAN_DIR = DATA_DIR / "clean"
CLEAN_DIR.mkdir(exist_ok=True)
NA_DAYS = 365243
PREV_DAYS = ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION"]
KEEP, EMPTY = "giữ", "→ trống"

s = pd.read_csv(DATA_DIR / "split.csv")
ids = set(s.SK_ID_CURR[s.tap == "train"])


def read_raw(name):
    return pd.read_csv(RAW_DIR / f"{name}.csv", dtype={"STATUS": "category"} if name == "bureau_balance" else None)


def train_rows(d):
    return d[d.SK_ID_CURR.isin(ids)].reset_index(drop=True)


def row(table, name, m, action=KEEP):
    print(f"{table:22s} {name:68s} {int(m.sum()):>10,} {m.mean():>9.4%}  {action}")

## Quy tắc làm sạch

In [2]:
def bureau_bad_dates(b):
    # Ngày kết thúc / cập nhật trước ngày mở khoản là không thể; ngày cập nhật sau ngày nộp là thông tin tương lai
    return {
        "DAYS_CREDIT_ENDDATE": b.DAYS_CREDIT_ENDDATE < b.DAYS_CREDIT,
        "DAYS_ENDDATE_FACT": b.DAYS_ENDDATE_FACT < b.DAYS_CREDIT,
        "DAYS_CREDIT_UPDATE": (b.DAYS_CREDIT_UPDATE < b.DAYS_CREDIT) | (b.DAYS_CREDIT_UPDATE > 0),
    }


def clean_application(a):
    a = a.copy().assign(FLAG_NOT_EMPLOYED=(a.DAYS_EMPLOYED == NA_DAYS).astype("int8"))
    a["DAYS_EMPLOYED"] = a.DAYS_EMPLOYED.replace(NA_DAYS, np.nan)
    a["NAME_FAMILY_STATUS"] = a.NAME_FAMILY_STATUS.replace("Unknown", np.nan)
    return a


def clean_bureau(b):
    for c, m in bureau_bad_dates(b).items():
        b[c] = b[c].mask(m)
    return b


def clean_previous(p):
    p["FLAG_NOT_TERMINATED"] = (p.DAYS_TERMINATION == NA_DAYS).astype("int8")  # khoản chưa kết thúc tại ngày nộp
    p[PREV_DAYS] = p[PREV_DAYS].replace(NA_DAYS, np.nan)
    return p


CLEAN = {"application_train": clean_application, "bureau": clean_bureau, "previous_application": clean_previous}


def clean(name, d):
    d = d.copy()
    for c in d.select_dtypes(["object", "str"]):
        d[c] = d[c].replace(["XNA", "XAP"], np.nan)
    return CLEAN.get(name, lambda x: x)(d)


def save(name, d):
    d.to_parquet(CLEAN_DIR / f"{name}.parquet", index=False)
    print(f"→ {name}.parquet: {len(d):,} dòng, {d.shape[1]} cột")

## Phát hiện ngoại lai (chỉ phát hiện)

Quy tắc 1,5 IQR: ngoài [Q1 − 1,5·IQR, Q3 + 1,5·IQR]. Cột có IQR = 0 (hơn nửa số dòng cùng một giá trị) thì mọi giá trị khác đều bị tính là ngoại lai,
nên được liệt kê riêng và không tính vào tỷ lệ hồ sơ có ngoại lai.

In [3]:
def outliers(table, d, target=None, top=8):
    cols = [c for c in d.select_dtypes("number") if not c.startswith(("SK_ID", "FLAG_", "TARGET")) and d[c].nunique() > 2]
    q1, q3 = d[cols].quantile(0.25), d[cols].quantile(0.75)
    m = (d[cols] < q1 - 1.5 * (q3 - q1)) | (d[cols] > q3 + 1.5 * (q3 - q1))
    zero = q3 == q1
    r = m.mean().sort_values(ascending=False)
    print(f"{table} (sau làm sạch): {len(cols)} cột số, cột có ngoại lai {(r > 0).sum()}, tỷ lệ trung vị {r.median():.2%}; "
          f"{zero.sum()} cột IQR = 0 ({', '.join(zero.index[zero])}); cao nhất:")
    for c in r.index[:top]:
        t = f"; TARGET trong ngoại lai {target[m[c]].mean():.4f}" if target is not None else ""
        print(f"    {c:32s} {r[c]:.2%} (IQR = {q3[c] - q1[c]:,.4g}; min {d[c].min():,.6g}, max {d[c].max():,.6g}){t}")
    return m.loc[:, ~zero].any(axis=1)

## 1. application

In [4]:
raw = read_raw("application_train")
a = train_rows(raw)
t = "application"
print(f"{t}: {len(a):,} hồ sơ train, TARGET=1 {a.TARGET.mean():.4f}")
row(t, "trùng SK_ID_CURR", a.SK_ID_CURR.duplicated())
row(t, "trùng toàn bộ cột trừ SK_ID_CURR", a.drop(columns="SK_ID_CURR").duplicated())
ne = a.DAYS_EMPLOYED == NA_DAYS
row(t, "DAYS_EMPLOYED = 365243 (không đi làm)", ne, EMPTY + " + cờ FLAG_NOT_EMPLOYED")
print(f"{'':22s}   nguồn thu nhập của nhóm này {a.NAME_INCOME_TYPE[ne].value_counts().to_dict()}; TARGET {a.TARGET[ne].mean():.4f}")
row(t, "ORGANIZATION_TYPE = XNA", a.ORGANIZATION_TYPE == "XNA", EMPTY)
row(t, "  trong đó không trùng nhóm DAYS_EMPLOYED = 365243", (a.ORGANIZATION_TYPE == "XNA") & ~ne)
row(t, "CODE_GENDER = XNA (cột bị bỏ theo QĐ-26)", a.CODE_GENDER == "XNA", EMPTY)
row(t, "NAME_FAMILY_STATUS = Unknown", a.NAME_FAMILY_STATUS == "Unknown", EMPTY)
emp = a.DAYS_EMPLOYED.mask(ne)
row(t, "cột DAYS_ có ngày sau ngày nộp (> 0)", (a.filter(like="DAYS_").drop(columns="DAYS_EMPLOYED") > 0).any(axis=1) | (emp > 0))
row(t, "bắt đầu công việc hiện tại trước 14 tuổi", emp < a.DAYS_BIRTH + 14 * 365.25)
row(t, "đăng ký cư trú hoặc cấp giấy tờ trước ngày sinh", (a.DAYS_REGISTRATION < a.DAYS_BIRTH) | (a.DAYS_ID_PUBLISH < a.DAYS_BIRTH))
row(t, "số con > số thành viên gia đình − 1", a.CNT_CHILDREN > a.CNT_FAM_MEMBERS - 1)
row(t, "không có xe nhưng có tuổi xe", (a.FLAG_OWN_CAR == "N") & a.OWN_CAR_AGE.notna())
row(t, "có xe nhưng thiếu tuổi xe", (a.FLAG_OWN_CAR == "Y") & a.OWN_CAR_AGE.isna())
row(t, "tiền trả mỗi kỳ > số tiền vay", a.AMT_ANNUITY > a.AMT_CREDIT)
row(t, "người quen quá hạn > người quen được quan sát (30, 60 ngày)",
    (a.DEF_30_CNT_SOCIAL_CIRCLE > a.OBS_30_CNT_SOCIAL_CIRCLE) | (a.DEF_60_CNT_SOCIAL_CIRCLE > a.OBS_60_CNT_SOCIAL_CIRCLE))
row(t, "hưu trí vẫn có ngày làm việc (QĐ-04 cho phép)", (a.NAME_INCOME_TYPE == "Pensioner") & ~ne)
row(t, "có nghề nghiệp nhưng DAYS_EMPLOYED = 365243", a.OCCUPATION_TYPE.notna() & ne)
share = {c: a[c].value_counts(normalize=True, dropna=False).iloc[0] for c in a}
print(f"{'':22s}   cột gần hằng số (một giá trị ≥ 99,9%, quyết định ở B6): {', '.join(f'{c} {v:.3%}' for c, v in share.items() if v >= 0.999)}")

application: 246,008 hồ sơ train, TARGET=1 0.0807
application            trùng SK_ID_CURR                                                              0   0.0000%  giữ


application            trùng toàn bộ cột trừ SK_ID_CURR                                              0   0.0000%  giữ
application            DAYS_EMPLOYED = 365243 (không đi làm)                                    44,143  17.9437%  → trống + cờ FLAG_NOT_EMPLOYED
                         nguồn thu nhập của nhóm này {'Pensioner': 44126, 'Unemployed': 17}; TARGET 0.0537
application            ORGANIZATION_TYPE = XNA                                                  44,143  17.9437%  → trống
application              trong đó không trùng nhóm DAYS_EMPLOYED = 365243                            0   0.0000%  giữ
application            CODE_GENDER = XNA (cột bị bỏ theo QĐ-26)                                      2   0.0008%  → trống
application            NAME_FAMILY_STATUS = Unknown                                                  2   0.0008%  → trống
application            cột DAYS_ có ngày sau ngày nộp (> 0)                                          0   0.0000%  giữ
application            bắt đ

                         cột gần hằng số (một giá trị ≥ 99,9%, quyết định ở B6): FLAG_MOBIL 100.000%, FLAG_DOCUMENT_2 99.996%, FLAG_DOCUMENT_4 99.991%, FLAG_DOCUMENT_7 99.981%, FLAG_DOCUMENT_10 99.998%, FLAG_DOCUMENT_12 100.000%, FLAG_DOCUMENT_17 99.975%, FLAG_DOCUMENT_19 99.939%, FLAG_DOCUMENT_20 99.950%, FLAG_DOCUMENT_21 99.968%


In [5]:
c = clean("application_train", a)
m = outliers(t, c, a.TARGET, top=10)
print(f"    hồ sơ có ngoại lai ở ít nhất một cột IQR > 0: {m.mean():.2%}; TARGET {a.TARGET[m].mean():.4f} so với {a.TARGET[~m].mean():.4f}")
for k in ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY"]:
    q1, q3 = c[k].quantile([0.25, 0.75])
    hi = c[k] > q3 + 1.5 * (q3 - q1)
    print(f"    {k}: ngoại lai phía trên {hi.mean():.2%}, TARGET {a.TARGET[hi].mean():.4f}")
big = a.nlargest(3, "AMT_INCOME_TOTAL")
print(f"    thu nhập lớn nhất {big.AMT_INCOME_TOTAL.tolist()} đơn vị (TARGET {big.TARGET.tolist()}); giữ")
assert c.DAYS_EMPLOYED.isna().sum() == ne.sum() and c.FLAG_NOT_EMPLOYED.sum() == ne.sum()
assert c.NAME_FAMILY_STATUS.isna().sum() == (a.NAME_FAMILY_STATUS == "Unknown").sum()
save("application_train", clean("application_train", raw))
del raw, a, c

application (sau làm sạch): 72 cột số, cột có ngoại lai 67, tỷ lệ trung vị 2.38%; 9 cột IQR = 0 (REGION_RATING_CLIENT, REGION_RATING_CLIENT_W_CITY, DEF_30_CNT_SOCIAL_CIRCLE, DEF_60_CNT_SOCIAL_CIRCLE, AMT_REQ_CREDIT_BUREAU_HOUR, AMT_REQ_CREDIT_BUREAU_DAY, AMT_REQ_CREDIT_BUREAU_WEEK, AMT_REQ_CREDIT_BUREAU_MON, AMT_REQ_CREDIT_BUREAU_QRT); cao nhất:
    REGION_RATING_CLIENT             26.17% (IQR = 0; min 1, max 3); TARGET trong ngoại lai 0.0862
    REGION_RATING_CLIENT_W_CITY      25.36% (IQR = 0; min 1, max 3); TARGET trong ngoại lai 0.0857
    AMT_REQ_CREDIT_BUREAU_QRT        16.42% (IQR = 0; min 0, max 19); TARGET trong ngoại lai 0.0713
    AMT_REQ_CREDIT_BUREAU_MON        14.19% (IQR = 0; min 0, max 27); TARGET trong ngoại lai 0.0738
    DEF_30_CNT_SOCIAL_CIRCLE         11.43% (IQR = 0; min 0, max 34); TARGET trong ngoại lai 0.1048
    DEF_60_CNT_SOCIAL_CIRCLE         8.35% (IQR = 0; min 0, max 24); TARGET trong ngoại lai 0.1091
    OBS_30_CNT_SOCIAL_CIRCLE         6.49% (IQR = 2; mi

→ application_train.parquet: 307,511 dòng, 123 cột


## 2. bureau, bureau_balance

In [6]:
raw = read_raw("bureau")
b = train_rows(raw)
t = "bureau"
print(f"{t}: {len(b):,} khoản của {b.SK_ID_CURR.nunique():,} khách train")
row(t, "trùng SK_ID_BUREAU", b.SK_ID_BUREAU.duplicated())
row(t, "trùng toàn bộ cột trừ SK_ID_BUREAU (dòng thừa)", b.drop(columns="SK_ID_BUREAU").duplicated())
bad = bureau_bad_dates(b)
row(t, "ngày kết thúc dự kiến trước ngày mở khoản", bad["DAYS_CREDIT_ENDDATE"], EMPTY)
row(t, "ngày kết thúc thực tế trước ngày mở khoản", bad["DAYS_ENDDATE_FACT"], EMPTY)
row(t, "ngày cập nhật trước ngày mở khoản", b.DAYS_CREDIT_UPDATE < b.DAYS_CREDIT, EMPTY)
row(t, "ngày cập nhật sau ngày nộp (thông tin tương lai)", b.DAYS_CREDIT_UPDATE > 0, EMPTY)
row(t, "  khách (trong số khách có bureau) có ít nhất một ngày bị làm trống", pd.concat(bad, axis=1).any(axis=1).groupby(b.SK_ID_CURR).any())
row(t, "  trong đó cách ngày nộp hơn 40.000 ngày (> 109 năm)", b.DAYS_CREDIT_ENDDATE < -40000, EMPTY)
cl, ac = b.CREDIT_ACTIVE == "Closed", b.CREDIT_ACTIVE == "Active"
row(t, "Closed nhưng ngày quá hạn > 0 (không dùng xếp nhóm, QĐ-07)", cl & (b.CREDIT_DAY_OVERDUE > 0))
row(t, "Closed nhưng dư nợ > 0", cl & (b.AMT_CREDIT_SUM_DEBT > 0))
row(t, "Closed nhưng thiếu ngày kết thúc thực tế", cl & b.DAYS_ENDDATE_FACT.isna())
row(t, "Active nhưng có ngày kết thúc thực tế", ac & b.DAYS_ENDDATE_FACT.notna())
row(t, "số tiền quá hạn > 0 nhưng số ngày quá hạn = 0", (b.AMT_CREDIT_SUM_OVERDUE > 0) & (b.CREDIT_DAY_OVERDUE == 0))
row(t, "số ngày quá hạn > 0 nhưng số tiền quá hạn = 0", (b.AMT_CREDIT_SUM_OVERDUE == 0) & (b.CREDIT_DAY_OVERDUE > 0))
row(t, "dư nợ âm (trả thừa)", b.AMT_CREDIT_SUM_DEBT < 0)
row(t, "dư nợ > số tiền cấp", b.AMT_CREDIT_SUM_DEBT > b.AMT_CREDIT_SUM + 1)
row(t, "tiền tệ khác currency 1 (số tiền đã quy đổi về một đồng tiền)", b.CREDIT_CURRENCY != "currency 1")
c = clean("bureau", b)
outliers(t, c)
for k, mk in bad.items():
    assert (c[k].isna() == (b[k].isna() | mk)).all()
all_bureau_ids, bureau_ids = set(raw.SK_ID_BUREAU), set(b.SK_ID_BUREAU)
save("bureau", clean("bureau", raw))
del raw, b, c

bureau: 1,171,299 khoản của 210,764 khách train
bureau                 trùng SK_ID_BUREAU                                                            0   0.0000%  giữ


bureau                 trùng toàn bộ cột trừ SK_ID_BUREAU (dòng thừa)                            1,275   0.1089%  giữ
bureau                 ngày kết thúc dự kiến trước ngày mở khoản                                   118   0.0101%  → trống
bureau                 ngày kết thúc thực tế trước ngày mở khoản                                    11   0.0009%  → trống
bureau                 ngày cập nhật trước ngày mở khoản                                            88   0.0075%  → trống
bureau                 ngày cập nhật sau ngày nộp (thông tin tương lai)                             15   0.0013%  → trống
bureau                   khách (trong số khách có bureau) có ít nhất một ngày bị làm trống         219   0.1039%  giữ
bureau                   trong đó cách ngày nộp hơn 40.000 ngày (> 109 năm)                        103   0.0088%  → trống
bureau                 Closed nhưng ngày quá hạn > 0 (không dùng xếp nhóm, QĐ-07)                   59   0.0050%  giữ
bureau                 Closed nhưng 

bureau (sau làm sạch): 12 cột số, cột có ngoại lai 10, tỷ lệ trung vị 3.01%; 5 cột IQR = 0 (CREDIT_DAY_OVERDUE, AMT_CREDIT_MAX_OVERDUE, CNT_CREDIT_PROLONG, AMT_CREDIT_SUM_LIMIT, AMT_CREDIT_SUM_OVERDUE); cao nhất:
    AMT_CREDIT_SUM_DEBT              16.34% (IQR = 3.96e+04; min -4.7056e+06, max 1.701e+08)
    AMT_CREDIT_SUM                   11.00% (IQR = 2.569e+05; min 0, max 1.701e+08)
    AMT_CREDIT_MAX_OVERDUE           7.29% (IQR = 0; min 0, max 1.15987e+08)
    DAYS_CREDIT_ENDDATE              4.55% (IQR = 1,617; min -2,890, max 31,199)
    AMT_CREDIT_SUM_LIMIT             4.44% (IQR = 0; min -401,347, max 4.7056e+06)
    DAYS_CREDIT_UPDATE               4.03% (IQR = 871; min -2,917, max 0)
    AMT_ANNUITY                      2.00% (IQR = 1.398e+04; min 0, max 5.74762e+07)
    CNT_CREDIT_PROLONG               0.60% (IQR = 0; min 0, max 9)


→ bureau.parquet: 1,716,428 dòng, 17 cột


In [7]:
bb = read_raw("bureau_balance")
t = "bureau_balance"
row(t, f"dòng không thuộc khoản nào trong bureau (toàn bảng {len(bb):,} dòng)", ~bb.SK_ID_BUREAU.isin(all_bureau_ids), "tự rơi khi nối")
save("bureau_balance", bb)
bb = bb[bb.SK_ID_BUREAU.isin(bureau_ids)].sort_values(["SK_ID_BUREAU", "MONTHS_BALANCE"])
print(f"{t}: {len(bb):,} dòng tháng của khoản bureau thuộc khách train")
row(t, "trùng (SK_ID_BUREAU, MONTHS_BALANCE)", bb.duplicated(["SK_ID_BUREAU", "MONTHS_BALANCE"]))
row(t, "tháng sau tháng nộp (MONTHS_BALANCE > 0)", bb.MONTHS_BALANCE > 0)
g = bb.groupby("SK_ID_BUREAU").MONTHS_BALANCE.agg(["min", "max", "size"])
row(t, "khoản có tháng bị hở giữa chuỗi", g["max"] - g["min"] + 1 != g["size"])
was_c = bb.STATUS.eq("C").groupby(bb.SK_ID_BUREAU).cummax().groupby(bb.SK_ID_BUREAU).shift(1, fill_value=False)
row(t, "tháng quá hạn (STATUS 1–5) sau khi đã tất toán (C)", was_c & bb.STATUS.isin(list("12345")))
row(t, "STATUS = X (không rõ, giữ làm một giá trị)", bb.STATUS == "X")
del bb, g, was_c

bureau_balance         dòng không thuộc khoản nào trong bureau (toàn bảng 27,299,925 dòng)   3,120,184  11.4293%  tự rơi khi nối


→ bureau_balance.parquet: 27,299,925 dòng, 3 cột


bureau_balance: 11,751,016 dòng tháng của khoản bureau thuộc khách train


bureau_balance         trùng (SK_ID_BUREAU, MONTHS_BALANCE)                                          0   0.0000%  giữ
bureau_balance         tháng sau tháng nộp (MONTHS_BALANCE > 0)                                      0   0.0000%  giữ
bureau_balance         khoản có tháng bị hở giữa chuỗi                                               0   0.0000%  giữ


bureau_balance         tháng quá hạn (STATUS 1–5) sau khi đã tất toán (C)                            2   0.0000%  giữ
bureau_balance         STATUS = X (không rõ, giữ làm một giá trị)                            2,266,724  19.2896%  giữ


## 3. previous_application

In [8]:
raw = read_raw("previous_application")
p = train_rows(raw)
t = "previous_application"
print(f"{t}: {len(p):,} hồ sơ cũ của {p.SK_ID_CURR.nunique():,} khách train")
row(t, "trùng SK_ID_PREV", p.SK_ID_PREV.duplicated())
dup = p.drop(columns="SK_ID_PREV").duplicated()
row(t, "trùng toàn bộ cột trừ SK_ID_PREV (dòng thừa)", dup)
print(f"{'':22s}   trạng thái của dòng thừa {p.NAME_CONTRACT_STATUS[dup].value_counts().to_dict()}; số tiền = 0 ở {(p.AMT_CREDIT[dup] == 0).mean():.2%}")
for col, v, name in [("FLAG_LAST_APPL_PER_CONTRACT", "N", "không phải hồ sơ cuối của hợp đồng (nộp nhầm)"), ("NFLAG_LAST_APPL_IN_DAY", 0, "không phải hồ sơ cuối trong ngày")]:
    mk = p[col] == v
    row(t, f"{col} = {v}: {name}", mk)
    print(f"{'':22s}   trạng thái {p.NAME_CONTRACT_STATUS[mk].value_counts().to_dict()}; trùng với dòng thừa {int((mk & dup).sum())}")
for col in PREV_DAYS:
    row(t, f"{col} = 365243", p[col] == NA_DAYS, EMPTY + (" + cờ FLAG_NOT_TERMINATED" if col == "DAYS_TERMINATION" else ""))
ap = p.NAME_CONTRACT_STATUS == "Approved"
row(t, "  DAYS_TERMINATION = 365243 ở hồ sơ không Approved", ~ap & (p.DAYS_TERMINATION == NA_DAYS))
row(t, "Approved nhưng mọi cột ngày đều trống (gốc, không phải 365243)", ap & p[PREV_DAYS].isna().all(axis=1))
unused = ap & (p.DAYS_FIRST_DUE.isna() | (p.DAYS_FIRST_DUE == NA_DAYS))
row(t, "Approved nhưng không có kỳ đến hạn đầu (khoản không được sử dụng)", unused)
row(t, "  trong đó DAYS_TERMINATION = 365243 (cờ chưa kết thúc = 1)", unused & (p.DAYS_TERMINATION == NA_DAYS))
print(f"{'':22s}   loại hợp đồng {p.NAME_CONTRACT_TYPE[unused & (p.DAYS_TERMINATION == NA_DAYS)].value_counts().to_dict()}")
for col in p.select_dtypes(["object", "str"]):
    for v in ["XNA", "XAP"]:
        if (p[col] == v).any():
            row(t, f"{col} = {v}", p[col] == v, EMPTY)
row(t, "Approved nhưng số tiền = 0 (hạn mức, xem bảng khớp)", ap & (p.AMT_CREDIT == 0))
row(t, "tiền trả trước âm", p.AMT_DOWN_PAYMENT < 0)
row(t, "ngày đến hạn cuối trước ngày đến hạn đầu", p.DAYS_LAST_DUE < p.DAYS_FIRST_DUE)
row(t, "ngày đến hạn đầu trước ngày quyết định", p.DAYS_FIRST_DUE < p.DAYS_DECISION)
c = clean("previous_application", p)
outliers(t, c)
assert (c[PREV_DAYS] != NA_DAYS).all().all() and c.FLAG_NOT_TERMINATED.sum() == (p.DAYS_TERMINATION == NA_DAYS).sum()
prev = p.set_index("SK_ID_PREV").SK_ID_CURR
save("previous_application", clean("previous_application", raw))
del raw, p, c

previous_application: 1,131,338 hồ sơ cũ của 232,834 khách train
previous_application   trùng SK_ID_PREV                                                              0   0.0000%  giữ


previous_application   trùng toàn bộ cột trừ SK_ID_PREV (dòng thừa)                             37,554   3.3194%  giữ
                         trạng thái của dòng thừa {'Canceled': 31751, 'Refused': 5418, 'Unused offer': 288, 'Approved': 97}; số tiền = 0 ở 90.41%
previous_application   FLAG_LAST_APPL_PER_CONTRACT = N: không phải hồ sơ cuối của hợp đồng (nộp nhầm)      5,780   0.5109%  giữ
                         trạng thái {'Refused': 5779, 'Canceled': 1}; trùng với dòng thừa 76
previous_application   NFLAG_LAST_APPL_IN_DAY = 0: không phải hồ sơ cuối trong ngày              4,000   0.3536%  giữ
                         trạng thái {'Refused': 3481, 'Approved': 511, 'Unused offer': 8}; trùng với dòng thừa 77
previous_application   DAYS_FIRST_DRAWING = 365243                                             639,369  56.5144%  → trống
previous_application   DAYS_FIRST_DUE = 365243                                                  27,278   2.4111%  → trống
previous_application   DAYS_LAST_DUE_1S

previous_application   NAME_PORTFOLIO = XNA                                                    245,699  21.7176%  → trống
previous_application   NAME_PRODUCT_TYPE = XNA                                                 720,630  63.6971%  → trống
previous_application   NAME_SELLER_INDUSTRY = XNA                                              571,858  50.5471%  → trống
previous_application   NAME_YIELD_GROUP = XNA                                                  343,334  30.3476%  → trống
previous_application   Approved nhưng số tiền = 0 (hạn mức, xem bảng khớp)                         940   0.0831%  giữ
previous_application   tiền trả trước âm                                                             1   0.0001%  giữ
previous_application   ngày đến hạn cuối trước ngày đến hạn đầu                                      0   0.0000%  giữ
previous_application   ngày đến hạn đầu trước ngày quyết định                                        0   0.0000%  giữ


previous_application (sau làm sạch): 17 cột số, cột có ngoại lai 13, tỷ lệ trung vị 0.53%; 0 cột IQR = 0 (); cao nhất:
    SELLERPLACE_AREA                 16.12% (IQR = 86; min -1, max 4e+06)
    AMT_APPLICATION                  12.37% (IQR = 1.602e+05; min 0, max 5.85e+06)
    AMT_CREDIT                       10.82% (IQR = 1.908e+05; min 0, max 4.50969e+06)
    AMT_GOODS_PRICE                  8.76% (IQR = 1.797e+05; min 0, max 5.85e+06)
    AMT_ANNUITY                      5.10% (IQR = 1.412e+04; min 0, max 418,058)
    AMT_DOWN_PAYMENT                 3.87% (IQR = 7,704; min -0.9, max 3.06004e+06)
    CNT_PAYMENT                      3.36% (IQR = 18; min 0, max 84)
    RATE_DOWN_PAYMENT                2.63% (IQR = 0.1089; min -1.36934e-05, max 1)


→ previous_application.parquet: 1,670,214 dòng, 38 cột


## 4. Ba bảng theo kỳ: POS_CASH_balance, credit_card_balance, installments_payments

In [9]:
def audit_monthly(name, d):
    t = name
    print(f"\n{t}: {len(d):,} dòng")
    owner = prev.reindex(d.SK_ID_PREV).values
    row(t, "SK_ID_PREV không có ở previous_application của khách train", pd.Series(pd.isna(owner)), "giữ (vẫn của khách đó)")
    row(t, "SK_ID_CURR lệch với previous_application", pd.Series(pd.notna(owner) & (owner != d.SK_ID_CURR.values)))
    row(t, "trùng toàn bộ dòng", d.duplicated())
    if name == "installments_payments":
        k = ["SK_ID_PREV", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER"]
        g = d.groupby(k).agg(h=("DAYS_INSTALMENT", "nunique"), a=("AMT_INSTALMENT", "nunique"))
        row(t, "kỳ có nhiều ngày đến hạn hoặc nhiều số phải trả", (g.h > 1) | (g.a > 1))
        row(t, "dòng chưa có lần trả (ngày trả, số trả trống)", d.AMT_PAYMENT.isna(), "giữ (kỳ chưa trả trước ngày nộp)")
        row(t, "số trả = 0", d.AMT_PAYMENT == 0)
        row(t, "số phải trả = 0", d.AMT_INSTALMENT == 0)
        row(t, "lần trả sau ngày nộp", d.DAYS_ENTRY_PAYMENT > 0)
        row(t, "phiên bản lịch = 0 (khoản hạn mức) thuộc khoản có ở credit_card_balance", d.SK_ID_PREV[d.NUM_INSTALMENT_VERSION == 0].isin(cc_prev))
        k = d[d.NUM_INSTALMENT_VERSION > 0].groupby(["SK_ID_PREV", "NUM_INSTALMENT_NUMBER", "DAYS_ENTRY_PAYMENT", "AMT_PAYMENT"]).NUM_INSTALMENT_VERSION.nunique()
        row(t, "lần trả (khoản, kỳ, ngày, số tiền) ghi ở nhiều phiên bản lịch", k > 1, "giữ (tránh cộng hai lần ở B6)")
    else:
        row(t, "trùng (SK_ID_PREV, MONTHS_BALANCE)", d.duplicated(["SK_ID_PREV", "MONTHS_BALANCE"]))
        row(t, "tháng sau tháng nộp (MONTHS_BALANCE ≥ 0)", d.MONTHS_BALANCE >= 0)
        row(t, "SK_DPD_DEF > SK_DPD", d.SK_DPD_DEF > d.SK_DPD)
        row(t, "NAME_CONTRACT_STATUS = XNA", d.NAME_CONTRACT_STATUS == "XNA", EMPTY)
    if name == "POS_CASH_balance":
        row(t, "số kỳ còn lại > tổng số kỳ", d.CNT_INSTALMENT_FUTURE > d.CNT_INSTALMENT)
    if name == "credit_card_balance":
        row(t, "số tiền rút âm", (d.filter(like="AMT_DRAWINGS") < 0).any(axis=1))
        row(t, "số tiền rút > 0 nhưng số lần rút = 0", (d.AMT_DRAWINGS_CURRENT > 0) & (d.CNT_DRAWINGS_CURRENT == 0))
        row(t, "dư nợ âm (trả thừa)", d.AMT_BALANCE < 0)
    outliers(t, d)


for name in ["POS_CASH_balance", "credit_card_balance", "installments_payments"]:
    raw = read_raw(name)
    d = train_rows(raw)
    if name == "credit_card_balance":
        cc_prev = set(d.SK_ID_PREV)
    audit_monthly(name, d)
    save(name, clean(name, raw))
    del raw, d


POS_CASH_balance: 6,834,611 dòng
POS_CASH_balance       SK_ID_PREV không có ở previous_application của khách train              232,037   3.3950%  giữ (vẫn của khách đó)
POS_CASH_balance       SK_ID_CURR lệch với previous_application                                      0   0.0000%  giữ


POS_CASH_balance       trùng toàn bộ dòng                                                            0   0.0000%  giữ


POS_CASH_balance       trùng (SK_ID_PREV, MONTHS_BALANCE)                                            0   0.0000%  giữ
POS_CASH_balance       tháng sau tháng nộp (MONTHS_BALANCE ≥ 0)                                      0   0.0000%  giữ
POS_CASH_balance       SK_DPD_DEF > SK_DPD                                                           0   0.0000%  giữ
POS_CASH_balance       NAME_CONTRACT_STATUS = XNA                                                    1   0.0000%  → trống
POS_CASH_balance       số kỳ còn lại > tổng số kỳ                                                5,232   0.0766%  giữ


POS_CASH_balance (sau làm sạch): 5 cột số, cột có ngoại lai 4, tỷ lệ trung vị 3.05%; 2 cột IQR = 0 (SK_DPD, SK_DPD_DEF); cao nhất:
    CNT_INSTALMENT_FUTURE            7.01% (IQR = 11; min 0, max 84)
    CNT_INSTALMENT                   5.01% (IQR = 14; min 1, max 84)
    SK_DPD                           3.05% (IQR = 0; min 0, max 4,231)
    SK_DPD_DEF                       1.17% (IQR = 0; min 0, max 3,373)
    MONTHS_BALANCE                   0.00% (IQR = 41; min -96, max -1)


→ POS_CASH_balance.parquet: 10,001,358 dòng, 8 cột


...

installments_payments  trùng toàn bộ dòng                                                            0   0.0000%  giữ


installments_payments  kỳ có nhiều ngày đến hạn hoặc nhiều số phải trả                               0   0.0000%  giữ
installments_payments  dòng chưa có lần trả (ngày trả, số trả trống)                             2,091   0.0225%  giữ (kỳ chưa trả trước ngày nộp)
installments_payments  số trả = 0                                                                1,007   0.0109%  giữ
installments_payments  số phải trả = 0                                                             135   0.0015%  giữ
installments_payments  lần trả sau ngày nộp                                                          0   0.0000%  giữ
installments_payments  phiên bản lịch = 0 (khoản hạn mức) thuộc khoản có ở credit_card_balance  2,774,727  99.9613%  giữ


installments_payments  lần trả (khoản, kỳ, ngày, số tiền) ghi ở nhiều phiên bản lịch            58,682   0.9106%  giữ (tránh cộng hai lần ở B6)


installments_payments (sau làm sạch): 6 cột số, cột có ngoại lai 4, tỷ lệ trung vị 5.58%; 0 cột IQR = 0 (); cao nhất:
    NUM_INSTALMENT_NUMBER            13.79% (IQR = 15; min 1, max 244)
    AMT_PAYMENT                      8.38% (IQR = 1.253e+04; min 0, max 3.77149e+06)
    AMT_INSTALMENT                   8.24% (IQR = 1.233e+04; min 0, max 3.77149e+06)
    NUM_INSTALMENT_VERSION           2.93% (IQR = 1; min 0, max 178)
    DAYS_ENTRY_PAYMENT               0.00% (IQR = 1,295; min -3,160, max -1)
    DAYS_INSTALMENT                  0.00% (IQR = 1,295; min -2,922, max -1)


→ installments_payments.parquet: 13,605,401 dòng, 8 cột
